In [ ]:
# Lab type: extend
# Course: DS202 — Time Series Analysis & Forecasting
# Lesson: Autocorrelation and Baseline Forecasts
# Task: The working baseline below implements the mean, naive, and seasonal-naive
#       forecasts from the lesson. Extend it with: (1) a drift baseline,
#       (2) the MASE metric, and (3) a second, shorter holdout. Skeletons provided.

# Lab: Extending the Baseline Suite

The lesson's three baselines are implemented and working below. You'll extend the
suite in three steps — each one is a tool practitioners actually reach for when the
basic three aren't enough.

**Outputs are cleared.** Run each cell to generate results.

## Setup and working baseline

In [ ]:
!pip install pandas numpy scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
days = pd.date_range("2023-01-01", "2025-12-31", freq="D")
t = np.arange(len(days))

trend   = 200 + 0.15 * t
weekday = np.array([-14, -18, -11, -6, 9, 52, 61])[days.dayofweek]
yearly  = 38 * np.sin(2 * np.pi * (days.dayofyear - 320) / 365.25)
noise   = rng.normal(0, 16, len(days))

orders = pd.Series(trend + weekday + yearly + noise, index=days, name="orders").round()
print(f"{len(orders)} days, {orders.index[0].date()} to {orders.index[-1].date()}")
orders.head()

In [ ]:
# Working baseline suite (from the lesson) — run as-is
train, test = orders.iloc[:-90], orders.iloc[-90:]
h = len(test)

def mae(fc):  return (test - fc).abs().mean()
def mape(fc): return ((test - fc).abs() / test).mean() * 100

mean_fc = pd.Series(np.repeat(train.mean(), h), index=test.index)
naive = pd.Series(np.repeat(train.iloc[-1], h), index=test.index)
seasonal_naive = pd.Series(np.tile(train.iloc[-7:].to_numpy(), h // 7 + 1)[:h], index=test.index)

results = {}
for name, fc in [("mean", mean_fc), ("naive", naive), ("seasonal naive", seasonal_naive)]:
    results[name] = fc
    print(f"{name:15s}  MAE {mae(fc):5.1f}   MAPE {mape(fc):4.1f}%")

## Extension 1: The drift baseline

The naive baseline predicts a flat line; the series trends. The **drift** baseline
draws a straight line through the first and last training observations and extends it:
forecast for step *i* is `last_value + i * (last - first) / (n - 1)`.

Implement it and add it to the comparison. Before running: predict where it will land
relative to `naive` and `seasonal naive`, and why.

In [ ]:
# TODO: implement the drift baseline
# slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
# drift = ...   (a Series indexed like test: last train value + steps * slope)

# print(f"drift            MAE {mae(drift):5.1f}   MAPE {mape(drift):4.1f}%")

<details>
<summary>🔑 Reveal model answer — Extension 1</summary>

```python
slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
steps = np.arange(1, h + 1)
drift = pd.Series(train.iloc[-1] + steps * slope, index=test.index)
print(f"drift            MAE {mae(drift):5.1f}   MAPE {mape(drift):4.1f}%")
```

Expected placement: better than `naive` (it follows the trend upward instead of
staying flat) but worse than `seasonal naive` (it has the trend but no weekly shape —
and the weekly swing on this series is larger than 90 days of trend drift). The
combination of both ideas is what real models like Holt-Winters deliver in Lesson 5.

</details>

## Extension 2: MASE — the metric with the baseline built in

MAPE needed a caveat about near-zero actuals. **MASE** (mean absolute scaled error)
sidesteps that and bakes the baseline comparison directly into the score: divide the
model's MAE by the MAE of the *seasonal naive computed one-step on the training set*.

`MASE < 1` means "beats the seasonal naive"; `> 1` means "loses to it".

Implement it: the scaling denominator is `train.diff(7).abs().mean()` (the average
error of predicting each training day from 7 days before). Then score all four
baselines with MASE.

In [ ]:
# TODO: implement MASE
# scale = ...          # in-sample one-step seasonal-naive MAE on train
# def mase(fc): ...
# for name, fc in results.items():
#     print(f"{name:15s}  MASE {mase(fc):5.2f}")

<details>
<summary>🔑 Reveal model answer — Extension 2</summary>

```python
scale = train.diff(7).abs().mean()

def mase(fc):
    return (test - fc).abs().mean() / scale

for name, fc in results.items():
    print(f"{name:15s}  MASE {mase(fc):5.2f}")
```

Note the subtlety: even the *seasonal naive itself* scores above 1 here, because the
denominator is a **one-step** seasonal naive on training data, while the forecast is
a 90-day-ahead tiled week — long horizons are genuinely harder. MASE's virtue is that
this comparison is explicit, unit-free, and safe when actuals approach zero (the
denominator comes from the training data, not the test actuals).

</details>

## Extension 3: How stable is the ranking?

A 90-day holdout covers the trending holiday quarter. Re-run the whole comparison
(all four baselines, MAE + MASE) on a **28-day holdout** instead. Then answer: which
baselines moved most, and what does that tell you about matching holdout length to
the decision the forecast serves?

In [ ]:
# TODO: rebuild train/test with a 28-day holdout and re-run the comparison
# train28, test28 = ...
# (reuse or refactor the code above — a small helper function is good practice)

<details>
<summary>🔑 Reveal model answer — Extension 3</summary>

```python
def evaluate(holdout):
    tr, te = orders.iloc[:-holdout], orders.iloc[-holdout:]
    scale = tr.diff(7).abs().mean()
    fcs = {
        "mean": pd.Series(np.repeat(tr.mean(), holdout), index=te.index),
        "naive": pd.Series(np.repeat(tr.iloc[-1], holdout), index=te.index),
        "seasonal naive": pd.Series(np.tile(tr.iloc[-7:].to_numpy(), holdout // 7 + 1)[:holdout], index=te.index),
        "drift": pd.Series(tr.iloc[-1] + np.arange(1, holdout + 1) * (tr.iloc[-1] - tr.iloc[0]) / (len(tr) - 1), index=te.index),
    }
    for name, fc in fcs.items():
        m = (te - fc).abs().mean()
        print(f"{name:15s}  MAE {m:5.1f}   MASE {m / scale:5.2f}")

print("— 90-day holdout —"); evaluate(90)
print("— 28-day holdout —"); evaluate(28)
```

On 28 days, `naive` and `drift` improve most: a flat (or gently sloped) line is far
less wrong over one month than over three, while `mean` stays terrible and
`seasonal naive` stays strong. The ranking of baselines depends on horizon — so the
holdout must match the horizon the business decision actually needs. A model chosen on
a 90-day backtest may be the wrong model for a 1-week operational forecast, and
vice versa (Lesson 7 formalises this with rolling-origin evaluation).

</details>

## Summary

> **Complete each sentence in one line.**

1. The drift baseline adds ________ to the naive forecast, but still lacks ________.
2. MASE scores a forecast relative to ________; below 1 means ________.
3. Baseline rankings change with holdout length, so the holdout must match ________.

<details>
<summary>🔑 Reveal summary answers</summary>

1. Drift adds **a linear trend (line through first and last training values)**, but
   still lacks **the seasonal shape**.
2. MASE scores relative to **the in-sample one-step seasonal naive**; below 1 means
   **the forecast beats that seasonal-naive floor**.
3. The holdout must match **the forecast horizon of the actual business decision**.

</details>